# GSM State Function

This notebook demonstrates the `GSMStateFn` class, a simplified approach to thermodynamic state function representation using symbolic variables organized into natural and conjugate categories.

## Import Required Libraries

Import sympy for symbolic mathematics, typing for type hints, and other necessary modules.

In [2]:
import sympy as sp
from IPython.display import Markdown, display
from bmcs_matmod.gsm_lagrange.core2.gsm_state_fn import GSMStateFn, StateFunctionType
from bmcs_matmod.gsm_lagrange.core2.gsm_vars import Scalar, markdown_vars_table
sp.init_printing(use_latex=True)

## Define Symbolic Variables

Create symbolic variables for our thermodynamic state function demonstration.

In [3]:
# Define Extended Test Model and Initialize GSMThermodynBox

# Define thermodynamic variables explicitly with descriptions and units
T = Scalar(r'\vartheta', 'T', 'Temperature (intensive)', unit='K', real=True, positive=True)
S = Scalar('S', 'S', 'Entropy (extensive)', unit='J⋅K⁻¹', real=True)
eps = Scalar(r'\varepsilon', 'eps', 'External strain (extensive)', real=True)
sig = Scalar(r'\sigma', 'sig', 'External stress (intensive)', unit='MPa', real=True)
omega = Scalar(r'\omega', 'omega', 'Damage parameter', real=True, positive=True)
Y = Scalar('Y', 'Y', 'Damage driving force', unit='J⋅m⁻³', real=True)

# Define material parameters explicitly with units
E = Scalar('E', 'E', "Young's modulus", unit='GPa', positive=True)
C_eps = Scalar(r'C_{\varepsilon}', 'C_eps', 'Thermal capacity', real=True, nonnegative=True)
T_0 = Scalar(r'\vartheta_0', 'T_0', 'Reference temperature', unit='K', real=True, nonnegative=True)

# Collect variables into lists for table rendering
thermodynamic_vars = [T, S, eps, sig, omega, Y]
material_parameters = [E, C_eps, T_0]

display(Markdown("### Thermodynamic Variables"))
thermo_table = markdown_vars_table(thermodynamic_vars)
display(Markdown(thermo_table))

display(Markdown("### Material Parameters"))
params_table = markdown_vars_table(material_parameters)
display(Markdown(params_table))

### Thermodynamic Variables

| Description | LaTeX Symbol | Codename | Base Unit | Unit | Rank | Dim | Key Assumptions |
|-------------|--------------|----------|-----------|------|------|-----|------------------|
| Temperature (intensive) | $\vartheta$ | `T` | K | K | 0 | 1d | positive, nonnegative, real, finite |
| Entropy (extensive) | $S$ | `S` | J⋅K⁻¹ | J⋅K⁻¹ | 0 | 1d | real, finite |
| External strain (extensive) | $\varepsilon$ | `eps` | 1 | 1 | 0 | 1d | real, finite |
| External stress (intensive) | $\sigma$ | `sig` | Pa | MPa | 0 | 1d | real, finite |
| Damage parameter | $\omega$ | `omega` | 1 | 1 | 0 | 1d | positive, nonnegative, real, finite |
| Damage driving force | $Y$ | `Y` | J⋅m⁻³ | J⋅m⁻³ | 0 | 1d | real, finite |


### Material Parameters

| Description | LaTeX Symbol | Codename | Base Unit | Unit | Rank | Dim | Key Assumptions |
|-------------|--------------|----------|-----------|------|------|-----|------------------|
| Young's modulus | $E$ | `E` | Pa | GPa | 0 | 1d | positive, nonnegative, real, finite |
| Thermal capacity | $C_{\varepsilon}$ | `C_eps` | J⋅K⁻¹ | J⋅K⁻¹ | 0 | 1d | nonnegative, real, finite |
| Reference temperature | $\vartheta_{0}$ | `T_0` | K | K | 0 | 1d | nonnegative, real, finite |


## Create Class Instance with Example Function

Instantiate the GSMStateFn class with a simple thermodynamic function expression - a Helmholtz free energy for a damaged elastic material.

In [4]:
# Define a Helmholtz free energy expression: F(T, ε, Ɛ)
# Simple damaged elastic energy with temperature dependence

# Extended Helmholtz free energy: elastic-damage + thermal capacity
F_elastic = sp.Rational(1, 2) * (1 - omega) * E * eps**2
F_thermal = C_eps * (T - T * sp.log(T / T_0))
F_expr = F_elastic + F_thermal

# Create GSMStateFn instance for Helmholtz free energy F(T, ε, Ɛ)
# Natural variables: T (thermal), eps (mechanical), Eps (internal)
# Conjugate variables: S (thermal), sig (mechanical), Sig (internal)

helmholtz_fn = GSMStateFn(
    fn_expr=F_expr,
    th_x_var=T,
    th_y_var=S,
    mc_x_var=eps,       # Strain is natural for Helmholtz (single symbol)
    mc_y_var=sig,       # Stress is conjugate (single symbol)
    Eps_var=omega,      # Internal natural variable (single symbol, not tuple)
    Sig_var=Y,          # Internal conjugate variable (single symbol, not tuple)
    state_function_type=StateFunctionType.HELMHOLTZ  # Required parameter
)

md_overview = helmholtz_fn.markdown_overview()
display(Markdown(md_overview))

# GSM State Function Overview

$F = C_{\varepsilon} \left(- \vartheta \log{\left(\frac{\vartheta}{\vartheta_{0}} \right)} + \vartheta\right) + E \varepsilon^{2} \left(\frac{1}{2} - \frac{\omega}{2}\right)$

## Natural Variables (independent)
| Description | LaTeX Symbol | Codename | Base Unit | Unit | Rank | Dim | Key Assumptions |
|-------------|--------------|----------|-----------|------|------|-----|------------------|
| Temperature (intensive) | $\vartheta$ | `T` | K | K | 0 | 1d | positive, nonnegative, real, finite |
| External strain (extensive) | $\varepsilon$ | `eps` | 1 | 1 | 0 | 1d | real, finite |
| Damage parameter | $\omega$ | `omega` | 1 | 1 | 0 | 1d | positive, nonnegative, real, finite |


## Conjugate Variables (derivatives)
| Description | LaTeX Symbol | Codename | Base Unit | Unit | Rank | Dim | Key Assumptions |
|-------------|--------------|----------|-----------|------|------|-----|------------------|
| Entropy (extensive) | $S$ | `S` | J⋅K⁻¹ | J⋅K⁻¹ | 0 | 1d | real, finite |
| External stress (intensive) | $\sigma$ | `sig` | Pa | MPa | 0 | 1d | real, finite |
| Damage driving force | $Y$ | `Y` | J⋅m⁻³ | J⋅m⁻³ | 0 | 1d | real, finite |


## Constitutive relations
$$ S = \frac{\partial f}{\partial \vartheta} = - C_{\varepsilon} \log{\left(\frac{\vartheta}{\vartheta_{0}} \right)} $$
$$ \sigma = \frac{\partial f}{\partial \varepsilon} = E \varepsilon \left(1 - \omega\right) $$
$$ Y = \frac{\partial f}{\partial \omega} = - \frac{E \varepsilon^{2}}{2} $$

## Expected Variable Organization
- **Natural:** ['T', 'eps', 'Eps']
- **Conjugate:** ['S', 'sig', 'Sig']
- **Thermally Intensive:** True
- **Mechanically Intensive:** False
- **Transformation Targets:** ['U', 'G', 'H']


## Test Variable Access

Demonstrate accessing individual variables from the class instance and verify their properties and types.

In [5]:
print("Variable Access Testing:")
print("=" * 30)

# Test individual variable access
print("Thermal Variables:")
print(f"  Natural (th_x_var): {helmholtz_fn.th_x_var} (type: {type(helmholtz_fn.th_x_var)})")
print(f"  Conjugate (th_y_var): {helmholtz_fn.th_y_var} (type: {type(helmholtz_fn.th_y_var)})")

print("\nMechanical Variables:")
print(f"  Natural (mc_x_var): {helmholtz_fn.mc_x_var} (type: {type(helmholtz_fn.mc_x_var)})")
print(f"  Conjugate (mc_y_var): {helmholtz_fn.mc_y_var} (type: {type(helmholtz_fn.mc_y_var)})")

print("\nInternal Variables:")
print(f"  Natural (Eps_var): {helmholtz_fn.Eps_var} (type: {type(helmholtz_fn.Eps_var)})")
print(f"  Conjugate (Sig_var): {helmholtz_fn.Sig_var} (type: {type(helmholtz_fn.Sig_var)})")

# Verify variables are sympy symbols
print(f"\nVerification - All variables are sympy.Symbol:")
all_vars = helmholtz_fn.get_all_variables()
for var in all_vars:
    print(f"  {var}: {isinstance(var, sp.Symbol)}")

Variable Access Testing:
Thermal Variables:
  Natural (th_x_var): \vartheta (type: <class 'bmcs_matmod.gsm_lagrange.core2.gsm_vars.Scalar'>)
  Conjugate (th_y_var): S (type: <class 'bmcs_matmod.gsm_lagrange.core2.gsm_vars.Scalar'>)

Mechanical Variables:
  Natural (mc_x_var): \varepsilon (type: <class 'bmcs_matmod.gsm_lagrange.core2.gsm_vars.Scalar'>)
  Conjugate (mc_y_var): \sigma (type: <class 'bmcs_matmod.gsm_lagrange.core2.gsm_vars.Scalar'>)

Internal Variables:
  Natural (Eps_var): \omega (type: <class 'bmcs_matmod.gsm_lagrange.core2.gsm_vars.Scalar'>)
  Conjugate (Sig_var): Y (type: <class 'bmcs_matmod.gsm_lagrange.core2.gsm_vars.Scalar'>)

Verification - All variables are sympy.Symbol:
  \vartheta: True
  \varepsilon: True
  \omega: True
  S: True
  \sigma: True
  Y: True


In [6]:
# Test the simplified interface - verify mechanical variables are single symbols
print("Testing simplified interface:")
print(f"Mechanical natural variable type: {type(helmholtz_fn.mc_x_var)}")
print(f"Mechanical conjugate variable type: {type(helmholtz_fn.mc_y_var)}")
print(f"Mechanical natural variable: {helmholtz_fn.mc_x_var}")
print(f"Mechanical conjugate variable: {helmholtz_fn.mc_y_var}")

print(f"\nAll natural variables: {helmholtz_fn.get_natural_variables()}")
print(f"All conjugate variables: {helmholtz_fn.get_conjugate_variables()}")

# Test mechanical constitutive relations
mech_relations = helmholtz_fn.get_mechanical_constitutive_relations()
print(f"\nMechanical constitutive relations: {mech_relations}")
print(f"Number of mechanical relations: {len(mech_relations)}")

Testing simplified interface:
Mechanical natural variable type: <class 'bmcs_matmod.gsm_lagrange.core2.gsm_vars.Scalar'>
Mechanical conjugate variable type: <class 'bmcs_matmod.gsm_lagrange.core2.gsm_vars.Scalar'>
Mechanical natural variable: \varepsilon
Mechanical conjugate variable: \sigma

All natural variables: [\vartheta, \varepsilon, \omega]
All conjugate variables: [S, \sigma, Y]

Mechanical constitutive relations: [(\sigma, E*\varepsilon*(1 - \omega))]
Number of mechanical relations: 1
